# Geemap


## 概述

本讲座介绍如何使用[Google Earth Engine](https://earthengine.google.com) (GEE) API结合[geemap](https://geemap.org) Python包进行基于云的地理空间分析。我们将涵盖Earth Engine的核心概念、可视化技术以及在Jupyter环境中执行分析的实用工作流。

## 学习目标

在本讲座结束时，您将能够：
* 解释Google Earth Engine平台的基本原理，包括其数据类型和基于云的功能
* 设置和配置geemap库，以便在云端进行地理空间分析
* 执行基本的地理空间操作，包括从Earth Engine过滤、可视化和导出数据
* 使用Earth Engine访问和处理栅格和矢量数据
* 从卫星图像创建延时动画
* 从Earth Engine数据创建交互式图表

### 前提条件

在开始使用geemap和Google Earth Engine之前，请确保完成以下步骤：

- **注册Google Earth Engine账户**：访问[https://code.earthengine.google.com/register](https://code.earthengine.google.com/register)注册GEE账户。注册后，您需要按照[此处](https://docs.google.com/document/d/1ZGSmrNm6_baqd8CHt33kIBWOlvkh-HLr46bODgJN1h0/edit?usp=sharing)的说明设置Google Cloud项目并启用Earth Engine API。GEE对[非商业和研究用途](https://earthengine.google.com/noncommercial)免费。

- **验证API认证**：要确认您的Earth Engine账户设置正确，请尝试在Google Colab中运行[此测试notebook](https://colab.research.google.com/github/gee-community/geemap/blob/master/docs/notebooks/geemap_colab.ipynb)。

## Google Earth Engine简介

### Google Earth Engine概述

[Google Earth Engine](https://earthengine.google.com)是一个云计算平台，支持大规模地理空间数据集的科学分析和可视化。它提供丰富的数据目录和处理能力，允许用户在全球范围内分析卫星图像和地理空间数据。

Earth Engine对[非商业和研究用途](https://earthengine.google.com/noncommercial)免费。非营利组织、研究机构、教育工作者、原住民政府和政府研究人员可以继续免费使用Earth Engine，这一政策已实施超过十年。但是，[商业用户](https://earthengine.google.com/commercial)可能需要付费许可。

### 安装Geemap

geemap包简化了在Python中使用Google Earth Engine的过程，提供了直观的API用于在Jupyter笔记本中进行可视化和分析。在Google Colab中，geemap已预先安装，但您可能需要为某些功能安装额外的依赖项。要安装带有可选依赖项的最新版本，请使用以下命令：

In [ ]:
# %pip install -U "geemap[workshop]"

### 导入库

首先，导入用于处理Google Earth Engine (GEE)和geemap的必要库。

In [ ]:
import ee
import geemap

### 认证和初始化Earth Engine

To authenticate and initialize your Earth Engine environment, you’ll need to create a Google Cloud Project and enable the [Earth Engine API](https://console.cloud.google.com/apis/api/earthengine.googleapis.com) for the project if you have not done so already. You can find detailed setup instructions [here](https://developers.google.com/earth-engine/guides/access#a-role-in-a-cloud-project).

In [ ]:
geemap.ee_initialize()

运行上述代码将提示您认证您的Earth Engine账户。按照说明完成Colab上的认证过程。为了将来避免此步骤，您可以设置一个名为`EARTHENGINE_TOKEN`的Colab密钥，并将您的Earth Engine令牌作为值，该令牌可以通过运行`geemap.get_ee_token()`获得。

In [ ]:
# geemap.get_ee_token()

### 创建交互式地图

Let’s create an interactive map using the `ipyleaflet` plotting backend. The `geemap.Map` class inherits from the `ipyleaflet.Map` class, so the syntax is similar to creating an interactive map with `ipyleaflet.Map`.

In [ ]:
m = geemap.Map()

要在Jupyter笔记本中显示地图，只需输入地图对象：

In [ ]:
m

To customize the map’s display, you can set various keyword arguments, such as `center` (latitude and longitude), `zoom`, `width`, and `height`. By default, the `width` is `100%`, filling the entire width of the Jupyter notebook cell. The `height` parameter can be a number (in pixels) or a string with a pixel format, e.g., `600px`.

#### 示例地图

**美国本土地图**

  要将地图居中显示在美国本土，请使用：

In [ ]:
m = geemap.Map(center=[40, -100], zoom=4, height="600px")
m

**田纳西州地图**

In [ ]:
m = geemap.Map(center=[35.746512, -86.209818], zoom=8)
m

**田纳西州诺克斯维尔市地图**

In [ ]:
m = geemap.Map(center=[35.959111, -83.909463], zoom=13)
m

要隐藏特定的地图控件，请将相应的控件参数设置为`False`，例如`draw_ctrl=False`来隐藏绘图控件。

In [ ]:
m = geemap.Map(data_ctrl=False, toolbar_ctrl=False, draw_ctrl=False)
m

### 添加底图

在geemap中添加底图有几种方法。您可以在创建地图时在`basemap`关键字参数中指定底图，或者使用`add_basemap`方法添加额外的底图图层。`Geemap`提供了数百个内置底图，使您可以用一行代码轻松地向地图添加图层。

要创建带有特定底图的地图，请使用如下所示的`basemap`参数。例如，`Esri.WorldImagery`提供了Esri全球影像底图。

In [ ]:
m = geemap.Map(basemap="Esri.WorldImagery")
m

#### 添加多个底图

您可以通过多次调用`add_basemap`来添加多个底图。例如，以下代码将`Esri.WorldTopoMap`和`OpenTopoMap`底图添加到现有地图：

In [ ]:
m.add_basemap("Esri.WorldTopoMap")

In [ ]:
m.add_basemap("OpenTopoMap")

#### 列出可用底图

要查看前10个可用底图，请使用：

In [ ]:
basemaps = list(geemap.basemaps.keys())
len(geemap.basemaps)

In [ ]:
basemaps[:10]

### Google底图

由于许可限制，Google底图默认不包含在geemap中。但是，用户可以根据自己的判断使用以下URL手动添加Google底图：

```text
    ROADMAP:  https://mt1.google.com/vt/lyrs=m&x={x}&y={y}&z={z}
    SATELLITE: https://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z={z}
    TERRAIN: https://mt1.google.com/vt/lyrs=p&x={x}&y={y}&z={z}
    HYBRID: https://mt1.google.com/vt/lyrs=y&x={x}&y={y}&z={z}
```

例如，要添加Google卫星影像作为切片图层：

In [ ]:
m = geemap.Map()
url = "https://mt1.google.com/vt/lyrs=y&x={x}&y={y}&z={z}"
m.add_tile_layer(url, name="Google Satellite", attribution="Google")
m

您还可以使用`add_text`方法向地图添加文本。文本将显示在地图上指定的位置。

In [ ]:
m.add_text(text="Hello from Earth Engine", position="bottomright")

## 交互式地图和工具简介

### 底图选择器

底图选择器允许您通过下拉菜单从各种底图中选择。将其添加到地图后，可以轻松访问不同的地图背景。

In [ ]:
m = geemap.Map()
m.add("basemap_selector")
m

### 图层管理器

图层管理器提供对图层可见性和透明度的控制。它允许切换图层的开启和关闭，并通过滑块调整透明度，使自定义地图视觉效果变得容易。

In [ ]:
m = geemap.Map(center=(40, -100), zoom=4)
dem = ee.Image("USGS/SRTMGL1_003")
states = ee.FeatureCollection("TIGER/2018/States")
vis_params = {
    "min": 0,
    "max": 4000,
    "palette": ["006633", "E5FFCC", "662A00", "D8D8D8", "F5F5F5"],
}
m.add_layer(dem, vis_params, "SRTM DEM")
m.add_layer(states, {}, "US States")
m.add("layer_manager")
m

### 检查器工具

检查器工具允许您在地图上点击以查询特定位置的Earth Engine数据。这有助于直接在地图上检查数据集的属性。

In [ ]:
m = geemap.Map(center=(40, -100), zoom=4)
dem = ee.Image("USGS/SRTMGL1_003")
landsat7 = ee.Image("LANDSAT/LE7_TOA_5YEAR/1999_2003")
states = ee.FeatureCollection("TIGER/2018/States")
vis_params = {
    "min": 0,
    "max": 4000,
    "palette": ["006633", "E5FFCC", "662A00", "D8D8D8", "F5F5F5"],
}
m.add_layer(dem, vis_params, "SRTM DEM")
m.add_layer(
    landsat7,
    {"bands": ["B4", "B3", "B2"], "min": 20, "max": 200, "gamma": 2.0},
    "Landsat 7",
)
m.add_layer(states, {}, "US States")
m.add("inspector")
m

### 图层编辑器

使用图层编辑器，您可以调整Earth Engine数据的可视化参数以获得更好的清晰度和焦点。它支持单波段图像、多波段图像和要素集合。

#### 单波段图像

In [ ]:
m = geemap.Map(center=(40, -100), zoom=4)
dem = ee.Image("USGS/SRTMGL1_003")
vis_params = {
    "min": 0,
    "max": 4000,
    "palette": ["006633", "E5FFCC", "662A00", "D8D8D8", "F5F5F5"],
}
m.add_layer(dem, vis_params, "SRTM DEM")
m.add("layer_editor", layer_dict=m.ee_layers["SRTM DEM"])
m

#### 多波段图像

In [ ]:
m = geemap.Map(center=(40, -100), zoom=4)
landsat7 = ee.Image("LANDSAT/LE7_TOA_5YEAR/1999_2003")
m.add_layer(
    landsat7,
    {"bands": ["B4", "B3", "B2"], "min": 20, "max": 200, "gamma": 2.0},
    "Landsat 7",
)
m.add("layer_editor", layer_dict=m.ee_layers["Landsat 7"])
m

#### 要素集合

In [ ]:
m = geemap.Map(center=(40, -100), zoom=4)
states = ee.FeatureCollection("TIGER/2018/States")
m.add_layer(states, {}, "US States")
m.add("layer_editor", layer_dict=m.ee_layers["US States"])
m

### 绘图控件

绘图控件功能允许您直接在地图上绘制形状，并自动将其转换为Earth Engine对象。按以下方式访问绘制的要素：

- 要将最后绘制的要素返回为`ee.Geometry()`，使用`m.user_roi`
- 要将所有绘制的要素返回为`ee.FeatureCollection()`，使用`m.user_rois`

In [ ]:
m = geemap.Map(center=(40, -100), zoom=4)
dem = ee.Image("USGS/SRTMGL1_003")
vis_params = {
    "min": 0,
    "max": 4000,
    "palette": "terrain",
}
m.add_layer(dem, vis_params, "SRTM DEM")
m.add("layer_manager")
m

In [ ]:
if m.user_roi is not None:
    image = dem.clip(m.user_roi)
    m.layers[1].visible = False
    m.add_layer(image, vis_params, "Clipped DEM")

## Earth Engine数据目录

[Earth Engine数据目录](https://developers.google.com/earth-engine/datasets)包含大量地理空间数据集。目前，该目录包含超过[1000个数据集](https://github.com/opengeos/Earth-Engine-Catalog/blob/master/gee_catalog.tsv)，总大小超过100 PB。著名的数据集包括Landsat、Sentinel、MODIS和NAIP。有关CSV或JSON格式的完整数据集列表，请参阅[Earth Engine数据集列表](https://github.com/giswqs/Earth-Engine-Catalog/blob/master/gee_catalog.tsv)。

### 在Earth Engine网站上搜索数据集

要直接在Earth Engine网站上浏览数据集：

- 查看完整目录：https://developers.google.com/earth-engine/datasets/catalog
- 按标签搜索：https://developers.google.com/earth-engine/datasets/tags

### 在Geemap中搜索数据集

[Earth Engine数据目录](https://developers.google.com/earth-engine/datasets/catalog)也可以直接从`geemap`中搜索。使用关键字按名称、标签或描述过滤数据集。例如，搜索"elevation"将只显示元数据中包含"elevation"的数据集，返回52个数据集。您可以滚动结果找到[NASA SRTM数字高程30m](https://developers.google.com/earth-engine/datasets/catalog/USGS_SRTMGL1_003#description)数据集。

In [ ]:
m = geemap.Map()
m

![](https://i.imgur.com/B3rf4QN.jpg)

Each dataset page contains detailed information such as availability, provider, Earth Engine snippet, tags, description, and example code. The Image, ImageCollection, or FeatureCollection ID is essential for accessing the dataset in Earth Engine’s JavaScript or Python APIs.

### 在Geemap中使用数据集模块

Geemap提供了一个内置的`datasets`模块，用于以编程方式访问特定数据集。例如，要访问USGS GAP阿拉斯加数据集：

In [ ]:
from geemap.datasets import DATA

In [ ]:
m = geemap.Map()
dataset = ee.Image(DATA.USGS_GAP_AK_2001)
m.add_layer(dataset, {}, "GAP Alaska")
m.centerObject(dataset, zoom=4)
m

要检索特定数据集的元数据，请使用`get_metadata`函数：

In [ ]:
from geemap.datasets import get_metadata

get_metadata(DATA.USGS_GAP_AK_2001)

## Earth Engine数据类型

Earth Engine objects are server-side entities, meaning they are stored and processed on Earth Engine’s servers rather than locally. This setup is comparable to streaming services (e.g., YouTube or Netflix) where the data remains on the provider’s servers, allowing us to access and process geospatial data in real time without downloading it to our local devices.

- **Image**：Earth Engine中的核心栅格数据类型，表示单个图像或场景。
- **ImageCollection**：图像的集合或序列，通常用于时间序列分析。
- **Geometry**：基本的矢量数据类型，包括点、线和多边形等形状。
- **Feature**：带有关联属性的Geometry，用于向矢量形状添加描述性数据。
- **FeatureCollection**：Feature的集合，类似于带有属性数据的shapefile。

## Earth Engine栅格数据

### 图像

在Earth Engine中，栅格数据表示为**Image**对象。每个Image由波段组成，每个波段都有自己的名称、数据类型、比例、掩码和投影。每个图像的元数据存储为一组属性。

#### 加载Earth Engine图像

要加载图像，请在`ee.Image`构造函数中使用Earth Engine资产ID。资产ID可以在Earth Engine数据目录中找到。例如，要加载NASA SRTM数字高程数据集：

In [ ]:
image = ee.Image("USGS/SRTMGL1_003")
image

#### 可视化Earth Engine图像

要可视化图像，您可以指定可视化参数，如最小值、最大值和颜色调色板。

In [ ]:
m = geemap.Map(center=[21.79, 70.87], zoom=3)
image = ee.Image("USGS/SRTMGL1_003")
vis_params = {
    "min": 0,
    "max": 6000,
    "palette": ["006633", "E5FFCC", "662A00", "D8D8D8", "F5F5F5"],  # 'terrain'
}
m.add_layer(image, vis_params, "SRTM")
m

### 图像集合

**ImageCollection**表示图像序列，通常用于时间数据，如卫星图像时间序列。ImageCollections通过将Earth Engine资产ID传递到`ImageCollection`构造函数来创建。资产ID可在[Earth Engine数据目录](https://developers.google.com/earth-engine/datasets)中找到。

#### 加载图像集合

要加载ImageCollection，例如Sentinel-2表面反射率集合：

In [ ]:
collection = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")

#### 过滤图像集合

您可以按位置和时间过滤ImageCollections。例如，要过滤覆盖特定位置且云量较少的图像：

In [ ]:
geometry = ee.Geometry.Point([-83.909463, 35.959111])
images = collection.filterBounds(geometry)
images.size()

In [ ]:
images.first()

In [ ]:
images = (
    collection.filterBounds(geometry)
    .filterDate("2024-07-01", "2024-10-01")
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 5))
)
images.size()

要在地图上查看过滤后的集合：

In [ ]:
m = geemap.Map()
image = images.first()

vis = {
    "min": 0.0,
    "max": 3000,
    "bands": ["B4", "B3", "B2"],
}

m.add_layer(image, vis, "Sentinel-2")
m.centerObject(image, 8)
m

#### 可视化图像集合

要将**ImageCollection**可视化为单个合成图像，您需要聚合该集合。例如，使用`collection.median()`方法创建一个表示集合中中值的图像。

In [ ]:
m = geemap.Map()
image = images.median()

vis = {
    "min": 0.0,
    "max": 3000,
    "bands": ["B8", "B4", "B3"],
}

m.add_layer(image, vis, "Sentinel-2")
m.centerObject(geometry, 8)
m

## Earth Engine矢量数据

A **FeatureCollection** is a collection of Features. It functions similarly to a GeoJSON FeatureCollection, where features include associated properties or attributes. For example, a shapefile’s data can be represented as a FeatureCollection.

### 加载要素集合

[Earth Engine数据目录](https://developers.google.com/earth-engine/datasets)包含各种矢量数据集，如美国人口普查数据和国家边界，作为要素集合。要素集合ID可通过搜索数据目录访问。例如，要加载美国人口普查局的[TIGER道路数据](https://developers.google.com/earth-engine/datasets/catalog/TIGER_2016_Roads)：

In [ ]:
m = geemap.Map()
fc = ee.FeatureCollection("TIGER/2016/Roads")
m.set_center(-83.909463, 35.959111, 12)
m.add_layer(fc, {}, "Census roads")
m

### 过滤要素集合

`filter`方法允许您根据特定属性值过滤FeatureCollection。例如，我们可以使用`eq`过滤器过滤特定州，选择"Tennessee"：

In [ ]:
m = geemap.Map()
states = ee.FeatureCollection("TIGER/2018/States")
fc = states.filter(ee.Filter.eq("NAME", "Tennessee"))
m.add_layer(fc, {}, "Tennessee")
m.center_object(fc, 7)
m

要检索集合中第一个要素的属性，请使用：

In [ ]:
feat = fc.first()
feat.toDictionary()

您还可以将FeatureCollection转换为Pandas DataFrame以便更容易分析：

In [ ]:
geemap.ee_to_df(fc)

In [ ]:
m = geemap.Map()
states = ee.FeatureCollection("TIGER/2018/States")
fc = states.filter(ee.Filter.inList("NAME", ["California", "Oregon", "Washington"]))
m.add_layer(fc, {}, "West Coast")
m.center_object(fc, 5)
m

In [ ]:
region = m.user_roi
if region is None:
    region = ee.Geometry.BBox(-88.40, 29.88, -77.90, 35.39)

fc = ee.FeatureCollection("TIGER/2018/States").filterBounds(region)
m.add_layer(fc, {}, "Southeastern U.S.")
m.center_object(fc, 6)

FeatureCollection可用于裁剪图像。`clipToCollection`方法将图像裁剪到要素集合的几何形状。例如，要将Landsat图像裁剪到法国边界：

In [ ]:
m = geemap.Map(center=(40, -100), zoom=4)
landsat7 = ee.Image("LANDSAT/LE7_TOA_5YEAR/1999_2003")
countries = ee.FeatureCollection("FAO/GAUL_SIMPLIFIED_500m/2015/level0")
fc = countries.filter(ee.Filter.eq("ADM0_NAME", "Germany"))
image = landsat7.clipToCollection(fc)
m.add_layer(
    image,
    {"bands": ["B4", "B3", "B2"], "min": 20, "max": 200, "gamma": 2.0},
    "Landsat 7",
)
m.center_object(fc, 6)
m

### 可视化要素集合

加载后，要素集合可以在交互式地图上可视化。例如，可视化人口普查数据中的美国州边界：

In [ ]:
m = geemap.Map(center=[40, -100], zoom=4)
states = ee.FeatureCollection("TIGER/2018/States")
m.add_layer(states, {}, "US States")
m

要素集合也可以使用额外参数进行样式化。要应用自定义样式，请指定颜色和线宽等选项：

In [ ]:
m = geemap.Map(center=[40, -100], zoom=4)
states = ee.FeatureCollection("TIGER/2018/States")
style = {"color": "0000ffff", "width": 2, "lineType": "solid", "fillColor": "FF000080"}
m.add_layer(states.style(**style), {}, "US States")
m

使用`add_styled_vector`，您可以应用颜色调色板按属性对不同要素进行样式化：

In [ ]:
m = geemap.Map(center=[40, -100], zoom=4)
states = ee.FeatureCollection("TIGER/2018/States")
vis_params = {
    "color": "000000",
    "colorOpacity": 1,
    "pointSize": 3,
    "pointShape": "circle",
    "width": 2,
    "lineType": "solid",
    "fillColorOpacity": 0.66,
}
palette = ["006633", "E5FFCC", "662A00", "D8D8D8", "F5F5F5"]
m.add_styled_vector(
    states, column="NAME", palette=palette, layer_name="Styled vector", **vis_params
)
m

## 更多可视化Earth Engine数据的工具

### 使用绘图工具

geemap中的绘图工具支持Earth Engine数据图层的可视化。在此示例中，添加了Landsat 7和Hyperion数据以及特定的可视化参数，允许比较这些数据集。然后添加绘图GUI以方便对数据进行详细探索。

In [ ]:
m = geemap.Map(center=[40, -100], zoom=4)

landsat7 = ee.Image("LANDSAT/LE7_TOA_5YEAR/1999_2003").select(
    ["B1", "B2", "B3", "B4", "B5", "B7"]
)

landsat_vis = {"bands": ["B4", "B3", "B2"], "gamma": 1.4}
m.add_layer(landsat7, landsat_vis, "Landsat")

hyperion = ee.ImageCollection("EO1/HYPERION").filter(
    ee.Filter.date("2016-01-01", "2017-03-01")
)

hyperion_vis = {
    "min": 1000.0,
    "max": 14000.0,
    "gamma": 2.5,
}
m.add_layer(hyperion, hyperion_vis, "Hyperion")
m.add_plot_gui()
m

设置Landsat的绘图选项，添加标记聚类和叠加层，增强地图上绘制数据的交互性。

In [ ]:
m.set_plot_options(add_marker_cluster=True, overlay=True)

Adjust the plotting options for Hyperion data by setting a bar plot type with marker clusters, suitable for visualizing Hyperion’s data values in bar format.

In [ ]:
m.set_plot_options(add_marker_cluster=True, plot_type="bar")

### 图例

#### 内置图例

Geemap提供内置图例，可以轻松添加到地图上，以更好地解释数据类别。在此，NLCD图例添加到WMS图层和Earth Engine土地覆盖图层，为数据的分类方案提供快速视觉参考。

In [ ]:
from geemap.legends import builtin_legends

打印所有可用的内置图例，可用于各种数据图层以增强地图可读性。

In [ ]:
for legend in builtin_legends:
    print(legend)

添加NLCD WMS图层及其相应的图例，该图例显示为叠加层，为用户提供信息丰富的图例显示。

In [ ]:
m = geemap.Map(center=[40, -100], zoom=4)
m.add_basemap("Esri.WorldImagery")
m.add_basemap("NLCD 2021 CONUS Land Cover")
m.add_legend(builtin_legend="NLCD", max_width="100px", height="455px")
m

添加NLCD土地覆盖的Earth Engine图层并显示其图例，指定标题、图例类型和尺寸，方便用户使用。

In [ ]:
m = geemap.Map(center=[40, -100], zoom=4)
m.add_basemap("Esri.WorldImagery")

nlcd = ee.Image("USGS/NLCD_RELEASES/2021_REL/NLCD/2021")
landcover = nlcd.select("landcover")

m.add_layer(landcover, {}, "NLCD Land Cover 2021")
m.add_legend(
    title="NLCD Land Cover Classification", builtin_legend="NLCD", height="455px"
)
m

#### 自定义图例

通过为每个类别定义唯一的标签和颜色来创建自定义图例，允许灵活的地图自定义。此示例使用颜色调色板标记多个地图类别。

In [ ]:
m = geemap.Map(add_google_map=False)

keys = ["One", "Two", "Three", "Four", "etc"]

# colors can be defined using either hex code or RGB (0-255, 0-255, 0-255)
colors = ["#8DD3C7", "#FFFFB3", "#BEBADA", "#FB8072", "#80B1D3"]
# legend_colors = [(255, 0, 0), (127, 255, 0), (127, 18, 25), (36, 70, 180), (96, 68 123)]

m.add_legend(keys=keys, colors=colors, position="bottomright")
m

使用字典定义自定义图例，该字典将特定颜色链接到标签。这种方法提供了灵活性来表示数据中的特定类别，例如各种土地覆盖类型。

In [ ]:
m = geemap.Map(center=[40, -100], zoom=4)
m.add_basemap("Esri.WorldImagery")

legend_dict = {
    "11 Open Water": "466b9f",
    "12 Perennial Ice/Snow": "d1def8",
    "21 Developed, Open Space": "dec5c5",
    "22 Developed, Low Intensity": "d99282",
    "23 Developed, Medium Intensity": "eb0000",
    "24 Developed High Intensity": "ab0000",
    "31 Barren Land (Rock/Sand/Clay)": "b3ac9f",
    "41 Deciduous Forest": "68ab5f",
    "42 Evergreen Forest": "1c5f2c",
    "43 Mixed Forest": "b5c58f",
    "51 Dwarf Scrub": "af963c",
    "52 Shrub/Scrub": "ccb879",
    "71 Grassland/Herbaceous": "dfdfc2",
    "72 Sedge/Herbaceous": "d1d182",
    "73 Lichens": "a3cc51",
    "74 Moss": "82ba9e",
    "81 Pasture/Hay": "dcd939",
    "82 Cultivated Crops": "ab6c28",
    "90 Woody Wetlands": "b8d9eb",
    "95 Emergent Herbaceous Wetlands": "6c9fb8",
}

nlcd = ee.Image("USGS/NLCD_RELEASES/2021_REL/NLCD/2021")
landcover = nlcd.select("landcover")

m.add_layer(landcover, {}, "NLCD Land Cover 2021")
m.add_legend(title="NLCD Land Cover Classification", legend_dict=legend_dict)
m

### 颜色条

添加表示高程数据的水平颜色条。此示例演示如何添加SRTM高程图层并叠加颜色条，以便快速视觉参考高程值。

In [ ]:
m = geemap.Map()

dem = ee.Image("USGS/SRTMGL1_003")
vis_params = {
    "min": 0,
    "max": 4000,
    "palette": ["006633", "E5FFCC", "662A00", "D8D8D8", "F5F5F5"],
}

m.add_layer(dem, vis_params, "SRTM DEM")
m.add_colorbar(vis_params, label="Elevation (m)", layer_name="SRTM DEM")
m

为高程数据添加垂直颜色条，调整其方向和尺寸以适应地图布局。

In [ ]:
m.add_colorbar(
    vis_params,
    label="Elevation (m)",
    layer_name="SRTM DEM",
    orientation="vertical",
    max_width="100px",
)

Make the color bar’s background transparent, which improves visual integration with the map when adding color bars for data layers.

In [ ]:
m.add_colorbar(
    vis_params,
    label="Elevation (m)",
    layer_name="SRTM DEM",
    orientation="vertical",
    max_width="100px",
    transparent_bg=True,
)

### 分屏地图

创建带有底图的分屏地图，允许用户并排比较两个不同的底图。

In [ ]:
m = geemap.Map()
m.split_map(left_layer="Esri.WorldTopoMap", right_layer="OpenTopoMap")
m

在分屏地图中使用Earth Engine图层比较2001年和2021年的NLCD数据。这种布局对于检查跨时间数据集之间的变化非常有效。

In [ ]:
m = geemap.Map(center=(40, -100), zoom=4, height=600)

nlcd_2001 = ee.Image("USGS/NLCD_RELEASES/2019_REL/NLCD/2001").select("landcover")
nlcd_2021 = ee.Image("USGS/NLCD_RELEASES/2021_REL/NLCD/2021").select("landcover")

left_layer = geemap.ee_tile_layer(nlcd_2001, {}, "NLCD 2001")
right_layer = geemap.ee_tile_layer(nlcd_2021, {}, "NLCD 2021")

m.split_map(left_layer, right_layer)
m

### 链接地图

设置一个2x2网格的链接地图，显示不同波段组合的Sentinel-2图像，非常适合比较多个视觉视角。请注意，此功能在Colab中可能无法工作。

In [ ]:
image = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterDate("2024-07-01", "2024-10-01")
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 5))
    .map(lambda img: img.divide(10000))
    .median()
)

vis_params = [
    {"bands": ["B4", "B3", "B2"], "min": 0, "max": 0.3, "gamma": 1.3},
    {"bands": ["B8", "B11", "B4"], "min": 0, "max": 0.3, "gamma": 1.3},
    {"bands": ["B8", "B4", "B3"], "min": 0, "max": 0.3, "gamma": 1.3},
    {"bands": ["B12", "B12", "B4"], "min": 0, "max": 0.3, "gamma": 1.3},
]

labels = [
    "Natural Color (B4/B3/B2)",
    "Land/Water (B8/B11/B4)",
    "Color Infrared (B8/B4/B3)",
    "Vegetation (B12/B11/B4)",
]

geemap.linked_maps(
    rows=2,
    cols=2,
    height="300px",
    center=[35.959111, -83.909463],
    zoom=12,
    ee_objects=[image],
    vis_params=vis_params,
    labels=labels,
    label_position="topright",
)

### 时间序列检查器

检索NLCD集合中可用的年份以设置时间序列。此步骤有助于确认检查随时间变化的可用时间点。

In [ ]:
m = geemap.Map(center=[40, -100], zoom=4)
collection = ee.ImageCollection("USGS/NLCD_RELEASES/2019_REL/NLCD").select("landcover")
vis_params = {"bands": ["landcover"]}
years = collection.aggregate_array("system:index").getInfo()
years

使用时间序列检查器比较不同年份NLCD数据的变化。此工具非常适合时间数据可视化，以交互格式显示土地覆盖如何变化。

In [ ]:
m.ts_inspector(
    left_ts=collection,
    right_ts=collection,
    left_names=years,
    right_names=years,
    left_vis=vis_params,
    right_vis=vis_params,
    width="80px",
)
m

### 时间滑块

创建时间滑块以探索MODIS植被数据，设置滑块以可视化特定时期的NDVI值。此功能允许用户逐月跟踪植被变化。

In [ ]:
m = geemap.Map()

collection = (
    ee.ImageCollection("MODIS/MCD43A4_006_NDVI")
    .filter(ee.Filter.date("2018-06-01", "2018-07-01"))
    .select("NDVI")
)
vis_params = {
    "min": 0.0,
    "max": 1.0,
    "palette": "ndvi",
}

m.add_time_slider(collection, vis_params, time_interval=2)
m

添加时间滑块以可视化24小时内的NOAA天气数据，使用颜色编码显示温度变化。滑块支持小时数据的时间探索。

In [ ]:
m = geemap.Map()

collection = (
    ee.ImageCollection("NOAA/GFS0P25")
    .filterDate("2018-12-22", "2018-12-23")
    .limit(24)
    .select("temperature_2m_above_ground")
)

vis_params = {
    "min": -40.0,
    "max": 35.0,
    "palette": ["blue", "purple", "cyan", "green", "yellow", "red"],
}

labels = [str(n).zfill(2) + ":00" for n in range(0, 24)]
m.add_time_slider(collection, vis_params, labels=labels, time_interval=1, opacity=0.8)
m

添加时间滑块以可视化带有特定波段和云量过滤的Sentinel-2图像。此功能支持图像数据的时间分析，允许用户探索季节性和其他随时间的变化。

In [ ]:
m = geemap.Map()

collection = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(ee.Geometry.Point([-83.909463, 35.959111]))
    .filterMetadata("CLOUDY_PIXEL_PERCENTAGE", "less_than", 10)
    .filter(ee.Filter.calendarRange(6, 8, "month"))
)

vis_params = {"min": 0, "max": 4000, "bands": ["B8", "B4", "B3"]}

m.add_time_slider(collection, vis_params)
m.set_center(-83.909463, 35.959111, 12)
m

## 矢量数据处理

### 从GeoJSON

将GeoJSON数据加载到Earth Engine FeatureCollection中。此示例从远程URL检索国家数据，将其转换为FeatureCollection并使用指定样式可视化。

In [ ]:
in_geojson = "https://github.com/gee-community/geemap/blob/master/examples/data/countries.geojson"
m = geemap.Map()
fc = geemap.geojson_to_ee(in_geojson)
m.add_layer(fc.style(**{"color": "ff0000", "fillColor": "00000000"}), {}, "Countries")
m

### 从Shapefile

下载并加载国家边界的shapefile。shapefile转换为FeatureCollection以便在地图上可视化。

In [ ]:
url = "https://github.com/gee-community/geemap/blob/master/examples/data/countries.zip"
geemap.download_file(url, overwrite=True)

In [ ]:
in_shp = "countries.shp"
fc = geemap.shp_to_ee(in_shp)

In [ ]:
m = geemap.Map()
m.add_layer(fc, {}, "Countries")
m

### 从GeoDataFrame

使用geopandas将shapefile读取到GeoDataFrame，然后将GeoDataFrame转换为Earth Engine FeatureCollection进行映射。

In [ ]:
import geopandas as gpd

gdf = gpd.read_file(in_shp)
fc = geemap.gdf_to_ee(gdf)

In [ ]:
m = geemap.Map()
m.add_layer(fc, {}, "Countries")
m

### 转换为GeoJSON

过滤美国州数据以选择田纳西州并将其保存为GeoJSON文件，该文件可以共享或在其他GIS工具中使用。

In [ ]:
m = geemap.Map()
states = ee.FeatureCollection("TIGER/2018/States")
fc = states.filter(ee.Filter.eq("NAME", "Tennessee"))
m.add_layer(fc, {}, "Tennessee")
m.center_object(fc, 7)
m

In [ ]:
geemap.ee_to_geojson(fc, filename="Tennessee.geojson")

### 转换为Shapefile

将过滤后的田纳西州FeatureCollection导出为shapefile格式，以便离线使用或与桌面GIS软件兼容。

In [ ]:
geemap.ee_to_shp(fc, filename="Tennessee.shp")

### 转换为GeoDataFrame

将Earth Engine FeatureCollection转换为GeoDataFrame，以便在Python中进一步分析或在交互式地图中使用。

In [ ]:
gdf = geemap.ee_to_gdf(fc)
gdf

In [ ]:
gdf.explore()

### 转换为DataFrame

将Earth Engine FeatureCollection转换为pandas DataFrame，然后可用于Python中的数据分析。

In [ ]:
df = geemap.ee_to_df(fc)
df

### 转换为CSV

将FeatureCollection数据导出为CSV文件，适用于电子表格应用程序和数据报告。

In [ ]:
geemap.ee_to_csv(fc, filename="Indiana.csv")

## 栅格数据处理

### 提取像素值

#### 提取值到点

加载并可视化SRTM DEM和Landsat 7图像。添加来自美国城市shapefile的点，工具将DEM值提取到这些点，并将结果保存到shapefile中。

In [ ]:
m = geemap.Map(center=[40, -100], zoom=4)

dem = ee.Image("USGS/SRTMGL1_003")
landsat7 = ee.Image("LANDSAT/LE7_TOA_5YEAR/1999_2003")

vis_params = {
    "min": 0,
    "max": 4000,
    "palette": ["006633", "E5FFCC", "662A00", "D8D8D8", "F5F5F5"],
}

m.add_layer(
    landsat7,
    {"bands": ["B4", "B3", "B2"], "min": 20, "max": 200, "gamma": 2},
    "Landsat 7",
)
m.add_layer(dem, vis_params, "SRTM DEM", True, 1)
m

In [ ]:
in_shp = "us_cities.shp"
url = "https://github.com/giswqs/data/raw/main/us/us_cities.zip"
geemap.download_file(url)

In [ ]:
in_fc = geemap.shp_to_ee(in_shp)
m.add_layer(in_fc, {}, "Cities")

In [ ]:
geemap.extract_values_to_points(in_fc, dem, out_fc="dem.shp")

In [ ]:
geemap.shp_to_gdf("dem.shp")

In [ ]:
geemap.extract_values_to_points(in_fc, landsat7, "landsat.csv")

In [ ]:
geemap.csv_to_df("landsat.csv")

#### 沿剖面提取像素值

使用地形底图可视化SRTM DEM。以交互方式或手动定义线剖面，并沿该线提取高程值。在线形图中显示高程剖面，然后将其导出为CSV文件以进行进一步分析。

In [ ]:
m = geemap.Map(center=[40, -100], zoom=4)
m.add_basemap("TERRAIN")

image = ee.Image("USGS/SRTMGL1_003")
vis_params = {
    "min": 0,
    "max": 4000,
    "palette": ["006633", "E5FFCC", "662A00", "D8D8D8", "F5F5F5"],
}
m.add_layer(image, vis_params, "SRTM DEM", True, 0.5)
m

In [ ]:
line = m.user_roi
if line is None:
    line = ee.Geometry.LineString(
        [[-120.2232, 36.3148], [-118.9269, 36.7121], [-117.2022, 36.7562]]
    )
    m.add_layer(line, {}, "ROI")
m.centerObject(line)

In [ ]:
reducer = "mean"
transect = geemap.extract_transect(
    image, line, n_segments=100, reducer=reducer, to_pandas=True
)
transect

In [ ]:
geemap.line_chart(
    data=transect,
    x="distance",
    y="mean",
    markers=True,
    x_label="Distance (m)",
    y_label="Elevation (m)",
    height=400,
)

In [ ]:
transect.to_csv("transect.csv")

### 分区统计

#### 图像和要素集合的分区统计

This section demonstrates the use of zonal statistics to calculate the mean elevation values within U.S. state boundaries using NASA’s SRTM DEM data and a 5-year Landsat composite. The `geemap.zonal_stats` function exports results to CSV files for analysis.

In [ ]:
m = geemap.Map(center=[40, -100], zoom=4)

# Add NASA SRTM
dem = ee.Image("USGS/SRTMGL1_003")
dem_vis = {
    "min": 0,
    "max": 4000,
    "palette": ["006633", "E5FFCC", "662A00", "D8D8D8", "F5F5F5"],
}
m.add_layer(dem, dem_vis, "SRTM DEM")

# Add 5-year Landsat TOA composite
landsat = ee.Image("LANDSAT/LE7_TOA_5YEAR/1999_2003")
landsat_vis = {"bands": ["B4", "B3", "B2"], "gamma": 1.4}
m.add_layer(landsat, landsat_vis, "Landsat", False)

# Add US Census States
states = ee.FeatureCollection("TIGER/2018/States")
style = {"fillColor": "00000000"}
m.add_layer(states.style(**style), {}, "US States")
m

In [ ]:
out_dem_stats = "dem_stats.csv"
geemap.zonal_stats(
    dem, states, out_dem_stats, statistics_type="MEAN", scale=1000, return_fc=False
)

In [ ]:
out_landsat_stats = "landsat_stats.csv"
geemap.zonal_stats(
    landsat,
    states,
    out_landsat_stats,
    statistics_type="MEAN",
    scale=1000,
    return_fc=False,
)

#### 按组分区统计

在此，分区统计应用于NLCD土地覆盖数据，计算每个美国州内每种土地覆盖类型的面积。结果以原始总数和百分比的形式保存到CSV文件中。这提供了对各州土地覆盖类别的空间分布的深入了解。

In [ ]:
m = geemap.Map(center=[40, -100], zoom=4)

# Add NLCD data
dataset = ee.Image("USGS/NLCD_RELEASES/2019_REL/NLCD/2019")
landcover = dataset.select("landcover")
m.add_layer(landcover, {}, "NLCD 2019")

# Add US census states
states = ee.FeatureCollection("TIGER/2018/States")
style = {"fillColor": "00000000"}
m.add_layer(states.style(**style), {}, "US States")

# Add NLCD legend
m.add_legend(title="NLCD Land Cover", builtin_legend="NLCD")
m

In [ ]:
nlcd_stats = "nlcd_stats.csv"

geemap.zonal_stats_by_group(
    landcover,
    states,
    nlcd_stats,
    statistics_type="SUM",
    denominator=1e6,
    decimal_places=2,
)

In [ ]:
nlcd_stats = "nlcd_stats_pct.csv"

geemap.zonal_stats_by_group(
    landcover,
    states,
    nlcd_stats,
    statistics_type="PERCENTAGE",
    denominator=1e6,
    decimal_places=2,
)

#### 两个图像的分区统计

此示例使用DEM和NLCD数据计算不同NLCD土地覆盖类型内的平均高程值。`geemap.image_stats_by_zone`函数提供汇总统计（例如均值和标准差），可以导出到CSV文件进行进一步分析或可视化。

In [ ]:
m = geemap.Map(center=[40, -100], zoom=4)
dem = ee.Image("USGS/3DEP/10m")
vis = {"min": 0, "max": 4000, "palette": "terrain"}
m.add_layer(dem, vis, "DEM")
m

In [ ]:
landcover = ee.Image("USGS/NLCD_RELEASES/2019_REL/NLCD/2019").select("landcover")
m.add_layer(landcover, {}, "NLCD 2019")
m.add_legend(title="NLCD Land Cover Classification", builtin_legend="NLCD")

In [ ]:
stats = geemap.image_stats_by_zone(dem, landcover, reducer="MEAN")
stats

In [ ]:
stats.to_csv("mean.csv", index=False)

In [ ]:
geemap.image_stats_by_zone(dem, landcover, out_csv="std.csv", reducer="STD")

### 地图代数

此示例通过计算5年Landsat合成的归一化差异植被指数(NDVI)和Landsat 8图像的增强植被指数(EVI)来演示基本的地图代数。这些指数用颜色刻度可视化，以突出植被区域。

In [ ]:
m = geemap.Map()

# Load a 5-year Landsat 7 composite 1999-2003.
landsat_1999 = ee.Image("LANDSAT/LE7_TOA_5YEAR/1999_2003")

# Compute NDVI.
ndvi_1999 = (
    landsat_1999.select("B4")
    .subtract(landsat_1999.select("B3"))
    .divide(landsat_1999.select("B4").add(landsat_1999.select("B3")))
)

vis = {"min": 0, "max": 1, "palette": "ndvi"}
m.add_layer(ndvi_1999, vis, "NDVI")
m.add_colorbar(vis, label="NDVI")
m

In [ ]:
# Load a Landsat 8 image.
image = ee.Image("LANDSAT/LC08/C02/T1_TOA/LC08_044034_20140318")

# Compute the EVI using an expression.
evi = image.expression(
    "2.5 * ((NIR - RED) / (NIR + 6 * RED - 7.5 * BLUE + 1))",
    {
        "NIR": image.select("B5"),
        "RED": image.select("B4"),
        "BLUE": image.select("B2"),
    },
)

# Define a map centered on San Francisco Bay.
m = geemap.Map(center=[37.4675, -122.1363], zoom=9)

vis = {"min": 0, "max": 1, "palette": "ndvi"}
m.add_layer(evi, vis, "EVI")
m.add_colorbar(vis, label="EVI")
m

## 处理本地地理空间数据

### 栅格数据

可以从本地文件加载单波段和多波段栅格数据。在此，加载数字高程模型(DEM)并使用地形颜色方案显示，另一个栅格以假彩色合成显示以突出不同特征。

#### 单波段栅格数据

In [ ]:
url = "https://github.com/giswqs/data/raw/main/raster/srtm90.tif"
filename = "dem.tif"
geemap.download_file(url, filename)

#### 多波段栅格数据

In [ ]:
m = geemap.Map()
m.add_raster(filename, cmap="terrain", layer_name="DEM")
vis_params = {"min": 0, "max": 4000, "palette": "terrain"}
m.add_colorbar(vis_params, label="Elevation (m)")
m

In [ ]:
url = "https://github.com/giswqs/data/raw/main/raster/cog.tif"
filename = "cog.tif"
geemap.download_file(url, filename)

In [ ]:
m = geemap.Map()
m.add_raster(filename, indexes=[4, 1, 2], layer_name="False color")
m

### 矢量数据

geemap可以加载和可视化各种矢量数据格式，包括GeoJSON、Shapefile、GeoDataFrame和GeoPackage格式。GeoJSON数据根据属性动态设置自定义颜色样式，而Shapefile和GeoDataFrame允许简单、结构化地向地图添加地理要素。

#### GeoJSON

In [ ]:
in_geojson = (
    "https://github.com/opengeos/datasets/releases/download/vector/cables.geojson"
)
m = geemap.Map()
m.add_geojson(in_geojson, layer_name="Cable lines", info_mode="on_hover")
m

In [ ]:
m = geemap.Map()
m.add_basemap("CartoDB.DarkMatter")
callback = lambda feat: {"color": feat["properties"]["color"], "weight": 2}
m.add_geojson(in_geojson, layer_name="Cable lines", style_callback=callback)
m

In [ ]:
url = "https://github.com/opengeos/datasets/releases/download/world/countries.geojson"
m = geemap.Map()
m.add_geojson(
    url, layer_name="Countries", fill_colors=["red", "yellow", "green", "orange"]
)
m

In [ ]:
import random

m = geemap.Map()


def random_color(feature):
    return {
        "color": "black",
        "weight": 3,
        "fillColor": random.choice(["red", "yellow", "green", "orange"]),
    }


m.add_geojson(url, layer_name="Countries", style_callback=random_color)
m

In [ ]:
m = geemap.Map()

style = {
    "stroke": True,
    "color": "#0000ff",
    "weight": 2,
    "opacity": 1,
    "fill": True,
    "fillColor": "#0000ff",
    "fillOpacity": 0.1,
}

hover_style = {"fillOpacity": 0.7}

m.add_geojson(url, layer_name="Countries", style=style, hover_style=hover_style)
m

#### Shapefile

In [ ]:
url = "https://github.com/opengeos/datasets/releases/download/world/countries.zip"
geemap.download_file(url, overwrite=True)

In [ ]:
m = geemap.Map()
in_shp = "countries.shp"
m.add_shp(in_shp, layer_name="Countries")
m

#### GeoDataFrame

In [ ]:
import geopandas as gpd

m = geemap.Map(center=[40, -100], zoom=4)
gdf = gpd.read_file("countries.shp")
m.add_gdf(gdf, layer_name="Countries")
m

#### GeoPackage

In [ ]:
m = geemap.Map()
data = "https://github.com/opengeos/datasets/releases/download/world/countries.gpkg"
m.add_vector(data, layer_name="Countries")
m

#### CSV转换为矢量

Geemap支持从CSV文件轻松转换为矢量数据格式，如GeoJSON、Shapefile和GeoPackage。CSV中的数据在地图上可视化，城市以按区域设置样式的标记显示，并添加图例以便清晰参考。

In [ ]:
data = "https://github.com/gee-community/geemap/blob/master/examples/data/us_cities.csv"
geemap.csv_to_df(data)

In [ ]:
geemap.csv_to_geojson(
    data, "cities.geojson", latitude="latitude", longitude="longitude"
)

In [ ]:
geemap.csv_to_shp(data, "cities.shp", latitude="latitude", longitude="longitude")

In [ ]:
geemap.csv_to_vector(data, "cities.gpkg", latitude="latitude", longitude="longitude")

In [ ]:
gdf = geemap.csv_to_gdf(data, latitude="latitude", longitude="longitude")
gdf

In [ ]:
cities = (
    "https://github.com/gee-community/geemap/blob/master/examples/data/us_cities.csv"
)
m = geemap.Map(center=[40, -100], zoom=4)
m.add_points_from_xy(cities, x="longitude", y="latitude")
m

In [ ]:
regions = "https://github.com/gee-community/geemap/blob/master/examples/data/us_regions.geojson"

In [ ]:
m = geemap.Map(center=[40, -100], zoom=4)
m.add_geojson(regions, layer_name="US Regions")
m.add_points_from_xy(
    cities,
    x="longitude",
    y="latitude",
    layer_name="US Cities",
    color_column="region",
    icon_names=["gear", "map", "leaf", "globe"],
    spin=True,
    add_legend=True,
)
m

In [ ]:
m = geemap.Map(center=[40, -100], zoom=4)
m.add_circle_markers_from_xy(
    data,
    x="longitude",
    y="latitude",
    radius=8,
    color="blue",
    fill_color="black",
    fill_opacity=0.5,
)
m

## 访问云优化GeoTIFF

### 云优化GeoTIFF (COG)

本节展示如何使用URL将云优化GeoTIFF (COG)图层添加到地图，这允许高效加载和可视化存储在云上的大型栅格文件。在此示例中，2020年加州野火的事件前图像添加到地图，允许无需本地文件存储进行遥感分析。

In [ ]:
m = geemap.Map()
url = "https://opendata.digitalglobe.com/events/california-fire-2020/pre-event/2018-02-16/pine-gulch-fire20/1030010076004E00.tif"
m.add_cog_layer(url, name="Fire (pre-event)")
m

In [ ]:
geemap.cog_center(url)

COG功能还允许检查数据，例如查看GeoTIFF文件中的边界和可用波段。可以在分屏地图视图中比较事件前和事件后图像，以分析火灾造成的变化。

In [ ]:
geemap.cog_bands(url)

In [ ]:
url2 = "https://opendata.digitalglobe.com/events/california-fire-2020/post-event/2020-08-14/pine-gulch-fire20/10300100AAC8DD00.tif"
m.add_cog_layer(url2, name="Fire (post-event)")

In [ ]:
m = geemap.Map(center=[39.4568, -108.5107], zoom=12)
m.split_map(left_layer=url2, right_layer=url)
m

### 时空资产目录 (STAC)

STAC集合可以类似地访问和可视化，支持时空数据的动态映射。通过指定STAC URL，您可以查看数据集边界、将地图居中在数据集上，并选择特定波段作为图层添加。此示例显示添加全色和假彩色图像以增强视觉分析。

In [ ]:
url = "https://tinyurl.com/22vptbws"

In [ ]:
geemap.stac_bounds(url)

In [ ]:
geemap.stac_center(url)

In [ ]:
geemap.stac_bands(url)

In [ ]:
m = geemap.Map()
m.add_stac_layer(url, bands=["pan"], name="Panchromatic")
m.add_stac_layer(url, bands=["B3", "B2", "B1"], name="False color")
m

## 导出Earth Engine数据

### 导出图像

This section demonstrates exporting Earth Engine images. First, a Landsat image is added to the map for visualization, and a rectangular region of interest (ROI) is specified. Exporting options include saving the image locally with specified scale and region or exporting to Google Drive. It’s possible to define custom CRS and transformation settings when saving the image.


向地图添加Landsat图像。

In [ ]:
m = geemap.Map()

image = ee.Image("LANDSAT/LC08/C02/T1_TOA/LC08_044034_20140318").select(
    ["B5", "B4", "B3"]
)

vis_params = {"min": 0, "max": 0.5, "gamma": [0.95, 1.1, 1]}

m.center_object(image)
m.add_layer(image, vis_params, "Landsat")
m

向地图添加矩形。

In [ ]:
region = ee.Geometry.BBox(-122.5955, 37.5339, -122.0982, 37.8252)
fc = ee.FeatureCollection(region)
style = {"color": "ffff00ff", "fillColor": "00000000"}
m.add_layer(fc.style(**style), {}, "ROI")

导出到本地驱动器

In [ ]:
geemap.ee_export_image(image, filename="landsat.tif", scale=30, region=region)

检查图像投影。

In [ ]:
projection = image.select(0).projection().getInfo()
projection

In [ ]:
crs = projection["crs"]
crs_transform = projection["transform"]

指定区域、crs和crs_transform。

In [ ]:
geemap.ee_export_image(
    image,
    filename="landsat_crs.tif",
    crs=crs,
    crs_transform=crs_transform,
    region=region,
)

导出到Google Drive

In [ ]:
geemap.ee_export_image_to_drive(
    image, description="landsat", folder="export", region=region, scale=30
)

In [ ]:
geemap.download_ee_image(image, "landsat.tif", scale=90)

### 导出图像集合

图像集合（如时间序列数据）可以过滤并导出为多个图像。在此，国家农业影像计划(NAIP)集合按日期和位置过滤。然后可以将集合保存到本地或发送到Google Drive，便于处理大型数据集。

In [ ]:
point = ee.Geometry.Point(-99.2222, 46.7816)
collection = (
    ee.ImageCollection("USDA/NAIP/DOQQ")
    .filterBounds(point)
    .filterDate("2008-01-01", "2018-01-01")
    .filter(ee.Filter.listContains("system:band_names", "N"))
)

In [ ]:
collection.aggregate_array("system:index")

导出到本地驱动器

In [ ]:
geemap.ee_export_image_collection(collection, out_dir=".", scale=10)

导出到Google Drive

In [ ]:
geemap.ee_export_image_collection_to_drive(collection, folder="export", scale=10)

### 导出要素集合

要素集合（如州边界）可以多种格式导出（例如Shapefile、GeoJSON和CSV）。此示例将阿拉斯加州边界以不同的矢量格式导出到本地和Google Drive。此外，导出的数据可以直接加载到GeoDataFrame中，以便在Python中进一步处理。

In [ ]:
m = geemap.Map()
states = ee.FeatureCollection("TIGER/2018/States")
fc = states.filter(ee.Filter.eq("NAME", "Alaska"))
m.add_layer(fc, {}, "Alaska")
m.center_object(fc, 4)
m

导出到本地驱动器

In [ ]:
geemap.ee_to_shp(fc, filename="Alaska.shp")

In [ ]:
geemap.ee_export_vector(fc, filename="Alaska.shp")

In [ ]:
geemap.ee_to_geojson(fc, filename="Alaska.geojson")

In [ ]:
geemap.ee_to_csv(fc, filename="Alaska.csv")

In [ ]:
gdf = geemap.ee_to_gdf(fc)
gdf

In [ ]:
df = geemap.ee_to_df(fc)
df

导出到Google Drive

In [ ]:
geemap.ee_export_vector_to_drive(
    fc, description="Alaska", fileFormat="SHP", folder="export"
)

## 创建延时动画

### Landsat延时

Landsat延时工具创建随时间变化的动画。在此，我们使用定义的感兴趣区域(ROI)为拉斯维加斯(1984-2023)生成延时GIF：

In [ ]:
m = geemap.Map()
roi = ee.Geometry.BBox(-115.5541, 35.8044, -113.9035, 36.5581)
m.add_layer(roi)
m.center_object(roi)
m

In [ ]:
timelapse = geemap.landsat_timelapse(
    roi,
    out_gif="las_vegas.gif",
    start_year=1984,
    end_year=2023,
    bands=["NIR", "Red", "Green"],
    frames_per_second=5,
    title="Las Vegas, NV",
    font_color="blue",
)
geemap.show_image(timelapse)

In [ ]:
m = geemap.Map()
roi = ee.Geometry.BBox(113.8252, 22.1988, 114.0851, 22.3497)
m.add_layer(roi)
m.center_object(roi)
m

In [ ]:
timelapse = geemap.landsat_timelapse(
    roi,
    out_gif="hong_kong.gif",
    start_year=1990,
    end_year=2022,
    start_date="01-01",
    end_date="12-31",
    bands=["SWIR1", "NIR", "Red"],
    frames_per_second=3,
    title="Hong Kong",
)
geemap.show_image(timelapse)

### Sentinel-2延时

此示例为选定的ROI生成Sentinel-2数据的延时动画，可以选择在地图上绘制自定义ROI。延时动画跨越2017-2023年，重点关注每年6月至9月期间。

In [ ]:
m = geemap.Map(center=[41.718934, -86.894547], zoom=12)
m

平移和缩放地图到感兴趣区域。使用绘图工具在地图上绘制矩形。如果未绘制矩形，将使用下面显示的默认矩形。

In [ ]:
roi = m.user_roi
if roi is None:
    roi = ee.Geometry.BBox(-87.0492, 41.6545, -86.7903, 41.7604)
    m.add_layer(roi)
    m.center_object(roi)

In [ ]:
timelapse = geemap.sentinel2_timelapse(
    roi,
    out_gif="sentinel2.gif",
    start_year=2017,
    end_year=2023,
    start_date="06-01",
    end_date="09-01",
    frequency="year",
    bands=["SWIR1", "NIR", "Red"],
    frames_per_second=3,
    title="Sentinel-2 Timelapse",
)
geemap.show_image(timelapse)

### MODIS延时

MODIS延时可以展示NDVI（植被指数）或海洋颜色数据随时间的变化。在植被示例中，我们可视化2000-2022年的NDVI并叠加国家边界。温度示例动画显示2018-2020年月温度趋势的Aqua卫星数据并叠加大陆边界。

In [ ]:
m = geemap.Map()
m

In [ ]:
roi = m.user_roi
if roi is None:
    roi = ee.Geometry.BBox(-18.6983, -36.1630, 52.2293, 38.1446)
    m.add_layer(roi)
    m.center_object(roi)

In [ ]:
timelapse = geemap.modis_ndvi_timelapse(
    roi,
    out_gif="ndvi.gif",
    data="Terra",
    band="NDVI",
    start_date="2000-01-01",
    end_date="2022-12-31",
    frames_per_second=3,
    title="MODIS NDVI Timelapse",
    overlay_data="countries",
)
geemap.show_image(timelapse)

MODIS温度

In [ ]:
m = geemap.Map()
m

In [ ]:
roi = m.user_roi
if roi is None:
    roi = ee.Geometry.BBox(-171.21, -57.13, 177.53, 79.99)
    m.add_layer(roi)
    m.center_object(roi)

In [ ]:
timelapse = geemap.modis_ocean_color_timelapse(
    satellite="Aqua",
    start_date="2018-01-01",
    end_date="2020-12-31",
    roi=roi,
    frequency="month",
    out_gif="temperature.gif",
    overlay_data="continents",
    overlay_color="yellow",
    overlay_opacity=0.5,
)
geemap.show_image(timelapse)

### GOES延时

GOES延时生成可以近实时地动画化大气现象。我们使用GOES-17数据为不同的ROI创建动画，包括飓风跟踪和火灾事件，使用自定义时间窗口和帧率。

In [ ]:
roi = ee.Geometry.BBox(167.1898, -28.5757, 202.6258, -12.4411)
start_date = "2022-01-15T03:00:00"
end_date = "2022-01-15T07:00:00"
data = "GOES-17"
scan = "full_disk"

In [ ]:
timelapse = geemap.goes_timelapse(
    roi, "goes.gif", start_date, end_date, data, scan, framesPerSecond=5
)
geemap.show_image(timelapse)

In [ ]:
roi = ee.Geometry.BBox(-159.5954, 24.5178, -114.2438, 60.4088)
start_date = "2021-10-24T14:00:00"
end_date = "2021-10-25T01:00:00"
data = "GOES-17"
scan = "full_disk"

In [ ]:
timelapse = geemap.goes_timelapse(
    roi, "hurricane.gif", start_date, end_date, data, scan, framesPerSecond=5
)
geemap.show_image(timelapse)

In [ ]:
roi = ee.Geometry.BBox(-121.0034, 36.8488, -117.9052, 39.0490)
start_date = "2020-09-05T15:00:00"
end_date = "2020-09-06T02:00:00"
data = "GOES-17"
scan = "full_disk"

In [ ]:
timelapse = geemap.goes_fire_timelapse(
    roi, "fire.gif", start_date, end_date, data, scan, framesPerSecond=5
)
geemap.show_image(timelapse)

## 图表

### 要素图表

#### 导入库

要为Earth Engine要素创建图表，导入所需的库，如`calendar`、`ee`、`geemap`和`chart`。

In [ ]:
import calendar
import ee
import geemap
from geemap import chart

使用`geemap.ee_initialize()`初始化Earth Engine。

In [ ]:
geemap.ee_initialize()

#### feature_by_feature

此函数允许沿x轴绘制要素，沿y轴表示选定属性的值。使用生态区数据绘制跨区域的月平均温度图表，其中月份作为系列列。

In [ ]:
ecoregions = ee.FeatureCollection("projects/google/charts_feature_example")
features = ecoregions.select("[0-9][0-9]_tmean|label")

In [ ]:
geemap.ee_to_df(features)

In [ ]:
x_property = "label"
y_properties = [str(x).zfill(2) + "_tmean" for x in range(1, 13)]

labels = calendar.month_abbr[1:]  # a list of month labels, e.g. ['Jan', 'Feb', ...]

colors = [
    "#604791",
    "#1d6b99",
    "#39a8a7",
    "#0f8755",
    "#76b349",
    "#f0af07",
    "#e37d05",
    "#cf513e",
    "#96356f",
    "#724173",
    "#9c4f97",
    "#696969",
]
title = "Average Monthly Temperature by Ecoregion"
x_label = "Ecoregion"
y_label = "Temperature"

In [ ]:
fig = chart.feature_by_feature(
    features,
    x_property,
    y_properties,
    colors=colors,
    labels=labels,
    title=title,
    x_label=x_label,
    y_label=y_label,
)
fig

![](https://i.imgur.com/MZa99Vf.png)

#### feature.by_property

显示每个生态区的月平均降水量。属性表示每个月的降水量值，允许按月跨区域进行视觉比较。

In [ ]:
ecoregions = ee.FeatureCollection("projects/google/charts_feature_example")
features = ecoregions.select("[0-9][0-9]_ppt|label")

In [ ]:
geemap.ee_to_df(features)

In [ ]:
keys = [str(x).zfill(2) + "_ppt" for x in range(1, 13)]
values = calendar.month_abbr[1:]  # a list of month labels, e.g. ['Jan', 'Feb', ...]

In [ ]:
x_properties = dict(zip(keys, values))
series_property = "label"
title = "Average Ecoregion Precipitation by Month"
colors = ["#f0af07", "#0f8755", "#76b349"]

In [ ]:
fig = chart.feature_by_property(
    features,
    x_properties,
    series_property,
    title=title,
    colors=colors,
    x_label="Month",
    y_label="Precipitation (mm)",
    legend_location="top-left",
)
fig

![](https://i.imgur.com/6RhuUc7.png)

#### feature_groups

根据属性值绘制要素组，显示分为"温暖"和"寒冷"类别的生态区1月平均温度。

In [ ]:
ecoregions = ee.FeatureCollection("projects/google/charts_feature_example")
features = ecoregions.select("[0-9][0-9]_ppt|label")

In [ ]:
features = ee.FeatureCollection("projects/google/charts_feature_example")
x_property = "label"
y_property = "01_tmean"
series_property = "warm"
title = "Average January Temperature by Ecoregion"
colors = ["#cf513e", "#1d6b99"]
labels = ["Warm", "Cold"]

In [ ]:
chart.feature_groups(
    features,
    x_property,
    y_property,
    series_property,
    title=title,
    colors=colors,
    x_label="Ecoregion",
    y_label="January Temperature (°C)",
    legend_location="top-right",
    labels=labels,
)

![](https://i.imgur.com/YFZlJtc.png)

#### feature_histogram

生成显示区域内7月降水量分布的直方图，显示不同降水量水平的像素计数，有助于理解数据分布。

In [ ]:
source = ee.ImageCollection("OREGONSTATE/PRISM/Norm91m").toBands()
region = ee.Geometry.Rectangle(-123.41, 40.43, -116.38, 45.14)
features = source.sample(region, 5000)

In [ ]:
geemap.ee_to_df(features.limit(5).select(["07_ppt"]))

In [ ]:
property = "07_ppt"
title = "July Precipitation Distribution for NW USA"

In [ ]:
fig = chart.feature_histogram(
    features,
    property,
    max_buckets=None,
    title=title,
    x_label="Precipitation (mm)",
    y_label="Pixel Count",
    colors=["#1d6b99"],
)
fig

![](https://i.imgur.com/ErIp7Oy.png)

### 图像图表

#### image_by_region

使用`image_by_region`函数显示每个生态区的月温度，聚合数据以可视化跨区域的月平均温度。

In [ ]:
ecoregions = ee.FeatureCollection("projects/google/charts_feature_example")
image = (
    ee.ImageCollection("OREGONSTATE/PRISM/Norm91m").toBands().select("[0-9][0-9]_tmean")
)

In [ ]:
labels = calendar.month_abbr[1:]  # a list of month labels, e.g. ['Jan', 'Feb', ...]
title = "Average Monthly Temperature by Ecoregion"

In [ ]:
fig = chart.image_by_region(
    image,
    ecoregions,
    reducer="mean",
    scale=500,
    x_property="label",
    title=title,
    x_label="Ecoregion",
    y_label="Temperature",
    labels=labels,
)
fig

![](https://i.imgur.com/y4rp3dK.png)

#### image_regions


生成显示跨区域月平均降水量的图表，使用属性表示月份并比较降水量水平。

In [ ]:
ecoregions = ee.FeatureCollection("projects/google/charts_feature_example")
image = (
    ee.ImageCollection("OREGONSTATE/PRISM/Norm91m").toBands().select("[0-9][0-9]_ppt")
)

In [ ]:
keys = [str(x).zfill(2) + "_ppt" for x in range(1, 13)]
values = calendar.month_abbr[1:]  # a list of month labels, e.g. ['Jan', 'Feb', ...]

In [ ]:
x_properties = dict(zip(keys, values))
title = "Average Ecoregion Precipitation by Month"
colors = ["#f0af07", "#0f8755", "#76b349"]

In [ ]:
fig = chart.image_regions(
    image,
    ecoregions,
    reducer="mean",
    scale=500,
    series_property="label",
    x_labels=x_properties,
    title=title,
    colors=colors,
    x_label="Month",
    y_label="Precipitation (mm)",
    legend_location="top-left",
)

![](https://i.imgur.com/5WJVCNY.png)

#### image_by_class

按波长显示生态区的光谱特征，显示跨光谱波段的反射率值以进行比较。

In [ ]:
ecoregions = ee.FeatureCollection("projects/google/charts_feature_example")

image = (
    ee.ImageCollection("MODIS/061/MOD09A1")
    .filter(ee.Filter.date("2018-06-01", "2018-09-01"))
    .select("sur_refl_b0[0-7]")
    .mean()
    .select([2, 3, 0, 1, 4, 5, 6])
)

wavelengths = [469, 555, 655, 858, 1240, 1640, 2130]

In [ ]:
fig = chart.image_by_class(
    image,
    class_band="label",
    region=ecoregions,
    reducer="MEAN",
    scale=500,
    x_labels=wavelengths,
    title="Ecoregion Spectral Signatures",
    x_label="Wavelength (nm)",
    y_label="Reflectance (x1e4)",
    colors=["#f0af07", "#0f8755", "#76b349"],
    legend_location="top-left",
    interpolation="basis",
)
fig

![](https://i.imgur.com/XqYHvBV.png)

#### image_histogram

生成直方图以可视化MODIS表面反射率在红色、近红外和短波红外波段的分布，提供按波段的数据分布洞察。

In [ ]:
image = (
    ee.ImageCollection("MODIS/061/MOD09A1")
    .filter(ee.Filter.date("2018-06-01", "2018-09-01"))
    .select(["sur_refl_b01", "sur_refl_b02", "sur_refl_b06"])
    .mean()
)

region = ee.Geometry.Rectangle([-112.60, 40.60, -111.18, 41.22])

In [ ]:
fig = chart.image_histogram(
    image,
    region,
    scale=500,
    max_buckets=200,
    min_bucket_width=1.0,
    max_raw=1000,
    max_pixels=int(1e6),
    title="MODIS SR Reflectance Histogram",
    labels=["Red", "NIR", "SWIR"],
    colors=["#cf513e", "#1d6b99", "#f0af07"],
)
fig

![](https://i.imgur.com/mY4yoYH.png)

### 图像集合图表

#### image_series

为森林区域创建植被指数(NDVI和EVI)的时间序列图表，有助于跟踪植被健康随时间的变化。

In [ ]:
# Define the forest feature collection.
forest = ee.FeatureCollection("projects/google/charts_feature_example").filter(
    ee.Filter.eq("label", "Forest")
)

# Load MODIS vegetation indices data and subset a decade of images.
veg_indices = (
    ee.ImageCollection("MODIS/061/MOD13A1")
    .filter(ee.Filter.date("2010-01-01", "2020-01-01"))
    .select(["NDVI", "EVI"])
)

In [ ]:
title = "Average Vegetation Index Value by Date for Forest"
x_label = "Year"
y_label = "Vegetation index (x1e4)"
colors = ["#e37d05", "#1d6b99"]

In [ ]:
fig = chart.image_series(
    veg_indices,
    region=forest,
    reducer=ee.Reducer.mean(),
    scale=500,
    x_property="system:time_start",
    chart_type="LineChart",
    x_cols="date",
    y_cols=["NDVI", "EVI"],
    colors=colors,
    title=title,
    x_label=x_label,
    y_label=y_label,
    legend_location="right",
)
fig

![](https://i.imgur.com/r9zSJh6.png)

#### image_series_by_region

按区域显示NDVI时间序列，比较沙漠、森林和草原区域。每个区域有一个独特的系列以便比较。

In [ ]:
# Import the example feature collection.
ecoregions = ee.FeatureCollection("projects/google/charts_feature_example")

# Load MODIS vegetation indices data and subset a decade of images.
veg_indices = (
    ee.ImageCollection("MODIS/061/MOD13A1")
    .filter(ee.Filter.date("2010-01-01", "2020-01-01"))
    .select(["NDVI"])
)

In [ ]:
title = "Average NDVI Value by Date"
x_label = "Date"
y_label = "NDVI (x1e4)"
x_cols = "index"
y_cols = ["Desert", "Forest", "Grassland"]
colors = ["#f0af07", "#0f8755", "#76b349"]

In [ ]:
fig = chart.image_series_by_region(
    veg_indices,
    regions=ecoregions,
    reducer=ee.Reducer.mean(),
    band="NDVI",
    scale=500,
    x_property="system:time_start",
    series_property="label",
    chart_type="LineChart",
    x_cols=x_cols,
    y_cols=y_cols,
    title=title,
    x_label=x_label,
    y_label=y_label,
    colors=colors,
    stroke_width=3,
    legend_location="bottom-left",
)
fig

![](https://i.imgur.com/rnILSfI.png)

#### image_doy_series

绘制特定区域的日序平均植被指数，显示十年内的季节性植被模式。

In [ ]:
# Import the example feature collection and subset the grassland feature.
grassland = ee.FeatureCollection("projects/google/charts_feature_example").filter(
    ee.Filter.eq("label", "Grassland")
)

# Load MODIS vegetation indices data and subset a decade of images.
veg_indices = (
    ee.ImageCollection("MODIS/061/MOD13A1")
    .filter(ee.Filter.date("2010-01-01", "2020-01-01"))
    .select(["NDVI", "EVI"])
)

In [ ]:
title = "Average Vegetation Index Value by Day of Year for Grassland"
x_label = "Day of Year"
y_label = "Vegetation Index (x1e4)"
colors = ["#f0af07", "#0f8755"]

In [ ]:
fig = chart.image_doy_series(
    image_collection=veg_indices,
    region=grassland,
    scale=500,
    chart_type="LineChart",
    title=title,
    x_label=x_label,
    y_label=y_label,
    colors=colors,
    stroke_width=5,
)
fig

![](https://i.imgur.com/F0z088e.png)

#### image_doy_series_by_year

比较两年的日序NDVI，显示2012年和2019年之间植被模式的变化。

In [ ]:
# Import the example feature collection and subset the grassland feature.
grassland = ee.FeatureCollection("projects/google/charts_feature_example").filter(
    ee.Filter.eq("label", "Grassland")
)

# Load MODIS vegetation indices data and subset years 2012 and 2019.
veg_indices = (
    ee.ImageCollection("MODIS/061/MOD13A1")
    .filter(
        ee.Filter.Or(
            ee.Filter.date("2012-01-01", "2013-01-01"),
            ee.Filter.date("2019-01-01", "2020-01-01"),
        )
    )
    .select(["NDVI", "EVI"])
)

In [ ]:
title = "Average Vegetation Index Value by Day of Year for Grassland"
x_label = "Day of Year"
y_label = "Vegetation Index (x1e4)"
colors = ["#e37d05", "#1d6b99"]

In [ ]:
fig = chart.doy_series_by_year(
    veg_indices,
    band_name="NDVI",
    region=grassland,
    scale=500,
    chart_type="LineChart",
    colors=colors,
    title=title,
    x_label=x_label,
    y_label=y_label,
    stroke_width=5,
)
fig

![](https://i.imgur.com/ui6zpbl.png)

#### image_doy_series_by_region

按区域可视化日序平均NDVI，展示沙漠、森林和草原生态区的季节性变化。

In [ ]:
# Import the example feature collection and subset the grassland feature.
ecoregions = ee.FeatureCollection("projects/google/charts_feature_example")

# Load MODIS vegetation indices data and subset a decade of images.
veg_indices = (
    ee.ImageCollection("MODIS/061/MOD13A1")
    .filter(ee.Filter.date("2010-01-01", "2020-01-01"))
    .select(["NDVI"])
)

In [ ]:
title = "Average Vegetation Index Value by Day of Year for Grassland"
x_label = "Day of Year"
y_label = "Vegetation Index (x1e4)"
colors = ["#f0af07", "#0f8755", "#76b349"]

In [ ]:
fig = chart.image_doy_series_by_region(
    veg_indices,
    "NDVI",
    ecoregions,
    region_reducer="mean",
    scale=500,
    year_reducer=ee.Reducer.mean(),
    start_day=1,
    end_day=366,
    series_property="label",
    stroke_width=5,
    chart_type="LineChart",
    title=title,
    x_label=x_label,
    y_label=y_label,
    colors=colors,
    legend_location="right",
)
fig

![](https://i.imgur.com/eGqGoRs.png)

### 数组和列表图表

#### 散点图

显示光谱波段之间的关系，绘制红色反射率与近红外和短波红外反射率的关系图，以检查波段相关性。

In [ ]:
# Import the example feature collection and subset the forest feature.
forest = ee.FeatureCollection("projects/google/charts_feature_example").filter(
    ee.Filter.eq("label", "Forest")
)

# Define a MODIS surface reflectance composite.
modisSr = (
    ee.ImageCollection("MODIS/061/MOD09A1")
    .filter(ee.Filter.date("2018-06-01", "2018-09-01"))
    .select("sur_refl_b0[0-7]")
    .mean()
)

# Reduce MODIS reflectance bands by forest region; get a dictionary with
# band names as keys, pixel values as lists.
pixel_vals = modisSr.reduceRegion(
    **{"reducer": ee.Reducer.toList(), "geometry": forest.geometry(), "scale": 2000}
)

# Convert NIR and SWIR value lists to an array to be plotted along the y-axis.
y_values = pixel_vals.toArray(["sur_refl_b02", "sur_refl_b06"])


# Get the red band value list; to be plotted along the x-axis.
x_values = ee.List(pixel_vals.get("sur_refl_b01"))

In [ ]:
title = "Relationship Among Spectral Bands for Forest Pixels"
colors = ["rgba(29,107,153,0.4)", "rgba(207,81,62,0.4)"]

In [ ]:
fig = chart.array_values(
    y_values,
    axis=1,
    x_labels=x_values,
    series_names=["NIR", "SWIR"],
    chart_type="ScatterChart",
    colors=colors,
    title=title,
    x_label="Red reflectance (x1e4)",
    y_label="NIR & SWIR reflectance (x1e4)",
    default_size=15,
    xlim=(0, 800),
)
fig

![](https://i.imgur.com/zkPlZIO.png)

In [ ]:
x = ee.List(pixel_vals.get("sur_refl_b01"))
y = ee.List(pixel_vals.get("sur_refl_b06"))

In [ ]:
fig = chart.array_values(
    y,
    x_labels=x,
    series_names=["SWIR"],
    chart_type="ScatterChart",
    colors=["rgba(207,81,62,0.4)"],
    title=title,
    x_label="Red reflectance (x1e4)",
    y_label="SWIR reflectance (x1e4)",
    default_size=15,
    xlim=(0, 800),
)
fig

![](https://i.imgur.com/WHUHjH6.png)

 #### 剖面线图

沿特定区域的剖面线绘制高程，提供地形高程的剖面视图。

In [ ]:
# Define a line across the Olympic Peninsula, USA.
transect = ee.Geometry.LineString([[-122.8, 47.8], [-124.5, 47.8]])

# Define a pixel coordinate image.
lat_lon_img = ee.Image.pixelLonLat()

# Import a digital surface model and add latitude and longitude bands.
elev_img = ee.Image("USGS/SRTMGL1_003").select("elevation").addBands(lat_lon_img)

# Reduce elevation and coordinate bands by transect line; get a dictionary with
# band names as keys, pixel values as lists.
elev_transect = elev_img.reduceRegion(
    reducer=ee.Reducer.toList(),
    geometry=transect,
    scale=1000,
)

# Get longitude and elevation value lists from the reduction dictionary.
lon = ee.List(elev_transect.get("longitude"))
elev = ee.List(elev_transect.get("elevation"))

# Sort the longitude and elevation values by ascending longitude.
lon_sort = lon.sort(lon)
elev_sort = elev.sort(lon)

In [ ]:
fig = chart.array_values(
    elev_sort,
    x_labels=lon_sort,
    series_names=["Elevation"],
    chart_type="AreaChart",
    colors=["#1d6b99"],
    title="Elevation Profile Across Longitude",
    x_label="Longitude",
    y_label="Elevation (m)",
    stroke_width=5,
    fill="bottom",
    fill_opacities=[0.4],
    ylim=(0, 2500),
)
fig

![](https://i.imgur.com/k3XRita.png)

#### 元数据散点图

通过绘制云量与几何RMSE的关系图来可视化Landsat图像元数据，有助于图像集合的质量评估。

In [ ]:
# Import a Landsat 8 collection and filter to a single path/row.
col = ee.ImageCollection("LANDSAT/LC08/C02/T1_L2").filter(
    ee.Filter.expression("WRS_PATH ==  45 && WRS_ROW == 30")
)

# Reduce image properties to a series of lists; one for each selected property.
propVals = col.reduceColumns(
    reducer=ee.Reducer.toList().repeat(2),
    selectors=["CLOUD_COVER", "GEOMETRIC_RMSE_MODEL"],
).get("list")

# Get selected image property value lists; to be plotted along x and y axes.
x = ee.List(ee.List(propVals).get(0))
y = ee.List(ee.List(propVals).get(1))

In [ ]:
colors = [geemap.hex_to_rgba("#96356f", 0.4)]
print(colors)

In [ ]:
fig = chart.array_values(
    y,
    x_labels=x,
    series_names=["RMSE"],
    chart_type="ScatterChart",
    colors=colors,
    title="Landsat 8 Image Collection Metadata (045030)",
    x_label="Cloud cover (%)",
    y_label="Geometric RMSE (m)",
    default_size=15,
)
fig

![](https://i.imgur.com/3COY3xd.png)

#### 映射函数散点和线图

将正弦函数绘制为线图，将数学函数映射到图表轴上进行可视化。

In [ ]:
import math

start = -2 * math.pi
end = 2 * math.pi
points = ee.List.sequence(start, end, None, 50)


def sin_func(val):
    return ee.Number(val).sin()


values = points.map(sin_func)

In [ ]:
fig = chart.array_values(
    values,
    points,
    chart_type="LineChart",
    colors=["#39a8a7"],
    title="Sine Function",
    x_label="radians",
    y_label="sin(x)",
    marker="circle",
)
fig

![](https://i.imgur.com/7qcxvey.png)

### 数据表图表

#### 手动数据表图表

从手动创建的数据表创建图表，例如美国州人口。

In [ ]:
import pandas as pd

In [ ]:
data = {
    "State": ["CA", "NY", "IL", "MI", "OR"],
    "Population": [37253956, 19378102, 12830632, 9883640, 3831074],
}

df = pd.DataFrame(data)
df

In [ ]:
fig = chart.Chart(
    df,
    x_cols=["State"],
    y_cols=["Population"],
    chart_type="ColumnChart",
    colors=["#1d6b99"],
    title="State Population (US census, 2010)",
    x_label="State",
    y_label="Population",
)
fig

![](https://i.imgur.com/vuxNmuh.png)

#### 计算数据表图表

从森林区域的MODIS植被指数数据计算数据表，然后绘制NDVI和EVI时间序列。

In [ ]:
# Import the example feature collection and subset the forest feature.
forest = ee.FeatureCollection("projects/google/charts_feature_example").filter(
    ee.Filter.eq("label", "Forest")
)

# Load MODIS vegetation indices data and subset a decade of images.
veg_indices = (
    ee.ImageCollection("MODIS/061/MOD13A1")
    .filter(ee.Filter.date("2010-01-01", "2020-01-01"))
    .select(["NDVI", "EVI"])
)

# Build a feature collection where each feature has a property that represents
# a DataFrame row.


def aggregate(img):
    # Reduce the image to the mean of pixels intersecting the forest ecoregion.
    stat = img.reduceRegion(
        **{"reducer": ee.Reducer.mean(), "geometry": forest, "scale": 500}
    )

    # Extract the reduction results along with the image date.
    date = geemap.image_date(img)
    evi = stat.get("EVI")
    ndvi = stat.get("NDVI")

    # Make a list of observation attributes to define a row in the DataTable.
    row = ee.List([date, evi, ndvi])

    # Return the row as a property of an ee.Feature.
    return ee.Feature(None, {"row": row})


reduction_table = veg_indices.map(aggregate)

# Aggregate the 'row' property from all features in the new feature collection
# to make a server-side 2-D list (DataTable).
data_table_server = reduction_table.aggregate_array("row")

# Define column names and properties for the DataTable. The order should
# correspond to the order in the construction of the 'row' property above.
column_header = ee.List([["Date", "EVI", "NDVI"]])

# Concatenate the column header to the table.
data_table_server = column_header.cat(data_table_server)

In [ ]:
data_table = chart.DataTable(data_table_server, date_column="Date")
data_table.head()

In [ ]:
fig = chart.Chart(
    data_table,
    chart_type="LineChart",
    x_cols="Date",
    y_cols=["EVI", "NDVI"],
    colors=["#e37d05", "#1d6b99"],
    title="Average Vegetation Index Value by Date for Forest",
    x_label="Date",
    y_label="Vegetation index (x1e4)",
    stroke_width=3,
    legend_location="right",
)
fig

![](https://i.imgur.com/PWei7QC.png)

#### 区间图表

显示带方差的年度NDVI时间序列，显示年际植被变化和全年的变异性。

In [ ]:
# Define a point to extract an NDVI time series for.
geometry = ee.Geometry.Point([-121.679, 36.479])

# Define a band of interest (NDVI), import the MODIS vegetation index dataset,
# and select the band.
band = "NDVI"
ndvi_col = ee.ImageCollection("MODIS/061/MOD13Q1").select(band)

# Map over the collection to add a day of year (doy) property to each image.


def set_doy(img):
    doy = ee.Date(img.get("system:time_start")).getRelative("day", "year")
    # Add 8 to day of year number so that the doy label represents the middle of
    # the 16-day MODIS NDVI composite.
    return img.set("doy", ee.Number(doy).add(8))


ndvi_col = ndvi_col.map(set_doy)

# Join all coincident day of year observations into a set of image collections.
distinct_doy = ndvi_col.filterDate("2013-01-01", "2014-01-01")
filter = ee.Filter.equals(**{"leftField": "doy", "rightField": "doy"})
join = ee.Join.saveAll("doy_matches")
join_col = ee.ImageCollection(join.apply(distinct_doy, ndvi_col, filter))

# Calculate the absolute range, interquartile range, and median for the set
# of images composing each coincident doy observation group. The result is
# an image collection with an image representative per unique doy observation
# with bands that describe the 0, 25, 50, 75, 100 percentiles for the set of
# coincident doy images.


def cal_percentiles(img):
    doyCol = ee.ImageCollection.fromImages(img.get("doy_matches"))

    return doyCol.reduce(
        ee.Reducer.percentile([0, 25, 50, 75, 100], ["p0", "p25", "p50", "p75", "p100"])
    ).set({"doy": img.get("doy")})


comp = ee.ImageCollection(join_col.map(cal_percentiles))

# Extract the inter-annual NDVI doy percentile statistics for the
# point of interest per unique doy representative. The result is
# is a feature collection where each feature is a doy representative that
# contains a property (row) describing the respective inter-annual NDVI
# variance, formatted as a list of values.


def order_percentiles(img):
    stats = ee.Dictionary(
        img.reduceRegion(
            **{"reducer": ee.Reducer.first(), "geometry": geometry, "scale": 250}
        )
    )

    # Order the percentile reduction elements according to how you want columns
    # in the DataTable arranged (x-axis values need to be first).
    row = ee.List(
        [
            img.get("doy"),
            stats.get(band + "_p50"),
            stats.get(band + "_p0"),
            stats.get(band + "_p25"),
            stats.get(band + "_p75"),
            stats.get(band + "_p100"),
        ]
    )

    # Return the row as a property of an ee.Feature.
    return ee.Feature(None, {"row": row})


reduction_table = comp.map(order_percentiles)

# Aggregate the 'row' properties to make a server-side 2-D array (DataTable).
data_table_server = reduction_table.aggregate_array("row")

# Define column names and properties for the DataTable. The order should
# correspond to the order in the construction of the 'row' property above.
column_header = ee.List([["DOY", "median", "p0", "p25", "p75", "p100"]])

# Concatenate the column header to the table.
data_table_server = column_header.cat(data_table_server)

In [ ]:
df = chart.DataTable(data_table_server)
df.head()

In [ ]:
fig = chart.Chart(
    df,
    chart_type="IntervalChart",
    x_cols="DOY",
    y_cols=["p0", "p25", "median", "p75", "p100"],
    title="Annual NDVI Time Series with Inter-Annual Variance",
    x_label="Day of Year",
    y_label="Vegetation index (x1e4)",
    stroke_width=1,
    fill="between",
    fill_colors=["#b6d1c6", "#83b191", "#83b191", "#b6d1c6"],
    fill_opacities=[0.6] * 4,
    labels=["p0", "p25", "median", "p75", "p100"],
    display_legend=True,
    legend_location="top-right",
    ylim=(0, 10000),
)
fig

![](https://i.imgur.com/i8ZrGPR.png)

## 练习

### 练习1：可视化DEM数据

在[Earth Engine数据目录](https://developers.google.com/earth-engine/datasets)中找到DEM数据集，并将其裁剪到特定区域（例如您的国家、州或城市）。使用适当的颜色调色板显示它。例如，下面的示例地图显示科罗拉多州的DEM。

![](https://i.imgur.com/OLeSt7n.png)

### 练习2：使用Sentinel-2或Landsat创建无云合成

使用Sentinel-2或Landsat-9数据为您选择的区域创建特定年份的无云合成。

使用[Sentinel-2](https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S2_SR_HARMONIZED)或[Landsat-9数据](https://developers.google.com/earth-engine/datasets/catalog/landsat-9)为您选择的区域创建特定年份的无云合成。使用适当的波段组合在地图上显示图像。例如，下面的示例地图显示2021年科罗拉多州Sentinel-2图像的无云假彩色合成。

![](https://i.imgur.com/xkxpkS1.png)

### 练习3：可视化NAIP图像

使用[NAIP](https://developers.google.com/earth-engine/datasets/catalog/USDA_NAIP_DOQQ)图像为您选择的美国县创建无云图像。例如，下面的示例地图显示田纳西州诺克斯县NAIP图像的无云真彩色合成。请记住，不同州可能有同名的县，因此请确保为所选州选择正确的县。

![](https://i.imgur.com/iZSGqGS.png)

### 练习4：可视化流域边界

仅使用轮廓颜色（无填充颜色）可视化[USGS流域边界数据集](https://developers.google.com/earth-engine/datasets/catalog/USGS_WBD_2017_HUC04)。

![](https://i.imgur.com/PLlNFq3.png)

### 练习5：可视化土地覆盖变化

使用[USGS国家土地覆盖数据库](https://developers.google.com/earth-engine/datasets/catalog/USGS_NLCD_RELEASES_2019_REL_NLCD)和[美国人口普查州](https://developers.google.com/earth-engine/datasets/catalog/TIGER_2018_States)创建分屏地图，以可视化您选择的美国州的土地覆盖变化(2001-2019)。确保将NLCD图例添加到地图中。

![](https://i.imgur.com/Au7Q5Ln.png)

### 练习6：创建Landsat延时动画

使用Landsat数据生成延时动画，以显示选定区域随时间的变化。

![Spain](https://github.com/user-attachments/assets/f12839c0-1c30-404d-b0ab-0fa12ce12d24)

## 总结

本讲座介绍了**geemap**，这是一个旨在简化**Google Earth Engine (GEE)**环境中地理空间数据可视化和分析的Python库。Geemap允许用户在Python笔记本中交互式地使用GEE数据执行GIS和遥感任务。主要主题包括使用geemap设置GEE、从Earth Engine数据目录加载和处理数据，以及处理矢量和栅格数据。使用Landsat、Sentinel、MODIS和GOES数据创建动画延时的技术展示了动态景观监测，而交互式绘图工具（如散点图和时间序列）支持数据探索和趋势分析。讲座还涵盖了以多种格式（如GeoTIFF、Shapefile和CSV）导出图像和要素，以促进数据共享。总体而言，本次课程提供了对geemap功能的基础理解，使用户能够在Python环境中高效地可视化、分析和从地球观测数据中获取洞察。